In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Assignment").getOrCreate()

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore.csv


In [7]:
import pandas as pd

In [8]:
# upload Sample - Superstore.csv first via files.upload()
pdf = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
df = spark.createDataFrame(pdf)

df.printSchema()
df.show(5)

root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+--

In [9]:
df_clean = df.dropDuplicates(["Customer ID", "Order Date"])
df_clean.show()

+------+--------------+----------+----------+--------------+-----------+-------------+--------+-------------+----------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|Customer Name| Segment|      Country|            City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+--------+-------------+----------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|  1300|CA-2015-121391| 10/4/2015| 10/7/2015|   First Class|   AA-10315|   Alex Avila|Consumer|United States|   San Francisco|    California|      94109|   West|OFF-ST-10001590|Office Supplies|     Storage

In [11]:
from pyspark.sql import functions as F

result = (df.filter(F.col("Region") == "West")
            .groupBy("Category")
            .agg(F.avg("Sales").alias("avg_sale_amount")))
result.show()

+---------------+------------------+
|       Category|   avg_sale_amount|
+---------------+------------------+
|Office Supplies|  116.422376910912|
|      Furniture| 357.3023246110322|
|     Technology|420.68753255425725|
+---------------+------------------+



In [25]:
df.na.fill({"Ship Mode": "Unknown"}).select("Ship Mode").distinct().show()

+--------------+
|     Ship Mode|
+--------------+
|   First Class|
|      Same Day|
|  Second Class|
|Standard Class|
+--------------+



In [16]:
city_counts = df.groupBy("City").count().filter(F.col("count") > 100)
city_counts.show()

+-------------+-----+
|         City|count|
+-------------+-----+
|  Springfield|  163|
|       Dallas|  157|
| Philadelphia|  537|
|  Los Angeles|  747|
|San Francisco|  510|
|    San Diego|  170|
|      Detroit|  115|
|     Columbus|  222|
|      Chicago|  314|
|      Seattle|  428|
|New York City|  915|
|      Houston|  377|
| Jacksonville|  125|
+-------------+-----+



In [26]:
df2 = df.drop("Postal Code")
df3 = df2.withColumnRenamed("Order Date", "event_time")
# df remains unchanged; df2 and df3 are new DataFrames

In [21]:
q8 = df.filter((F.col("Sales").between(100, 500)) & (F.col("Segment") == "Consumer"))
q8.show(5)


+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+-------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name| Segment|      Country|         City|       State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+-------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|Consumer|United States|    Henderson|    Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Some

In [19]:
df_updated = (df.withColumn("event_time", F.to_timestamp(F.col("Order Date"), "M/d/yyyy"))
                .drop("Order Date"))
df_updated.select("event_time").show(5)


+-------------------+
|         event_time|
+-------------------+
|2016-11-08 00:00:00|
|2016-11-08 00:00:00|
|2016-06-12 00:00:00|
|2015-10-11 00:00:00|
|2015-10-11 00:00:00|
+-------------------+
only showing top 5 rows


In [22]:
q12 = df.filter(F.col("Customer ID").isNotNull() & (F.trim(F.col("Customer Name")) != ""))
q12.count()

9994

In [23]:
df.agg(F.min("Sales").alias("min_price"),
       F.max("Sales").alias("max_price"),
       F.mean("Sales").alias("mean_price")).show()

+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|    0.444| 22638.48|229.85800083049725|
+---------+---------+------------------+



In [24]:
pipeline_result = (df.dropDuplicates()
                     .na.fill({"Sales": 0})
                     .groupBy("Region")
                     .agg(F.sum("Sales").alias("total_revenue")))
pipeline_result.show()

+-------+------------------+
| Region|     total_revenue|
+-------+------------------+
|  South|        391721.905|
|Central|501239.89079999994|
|   East| 678781.2400000008|
|   West| 725457.8245000013|
+-------+------------------+

